# Solar Power Prediction (Pune, India)
**Candidate / Author:** [YOUR NAME]  
**Location:** Pune (Lat 18.52, Lon 73.86)  
**Tech Stack:** Python, pandas, numpy, scikit-learn, requests, plotly, joblib

---

### Project Overview
Predicting hourly solar power generation (kW) in Pune using weather data and a 48-hour forecast from Open-Meteo.

Steps:
1. **Data Ingestion:** Generate 60 days of hourly weather and solar data (Jan 1 - Feb 29, 2024).
2. **EDA & Cleaning:** Check the data, handle missing values, and clip to valid ranges.
3. **Feature Engineering:** Add hour, sin/cos of hour, day of week, and weekend flag.
4. **Train / Val Split:** First 50 days to train, last 10 days to validate (no shuffling).
5. **Model Training:** Train a LinearRegression baseline and tune a RandomForest model.
6. **Weather API:** Fetch 48-hour Pune forecast from Open-Meteo (free, no API key).
7. **Prediction & Plot:** Predict future solar output, clip negatives to 0, and plot history + forecast with Plotly.

In [1]:
# Cell 2: Imports
import pandas as pd
import numpy as np
import joblib
from datetime import datetime, timedelta

# Plotly for interactive data visualization
import plotly.graph_objects as go
import plotly.express as px

# Import project modules
from preprocess import load_and_clean_data, add_features
from train import train_and_save_model
from weather_api import get_pune_weather_forecast

print("All libraries and modular functions imported successfully!")

All libraries and modular functions imported successfully!


### 1. Generate Dataset
We generate 60 days (1440 hours) of hourly weather and solar data for Pune from 1 Jan to 29 Feb 2024 using the script from the assignment.

In [2]:
np.random.seed(42)  # For reproducibility

rows = []
start = datetime(2024, 1, 1)

for i in range(24 * 60):  # 60 days * 24 hours = 1440 rows
    ts = start + timedelta(hours=i)
    hour = ts.hour
    temperature = 18 + 10 * np.sin((hour / 24) * 2 * np.pi) + np.random.normal(0, 1)
    humidity = np.clip(60 + np.random.normal(0, 10), 20, 100)
    cloud_cover = np.clip(np.random.normal(40, 25), 0, 100)
    wind_speed = np.clip(np.random.normal(5, 2), 0, 15)
    precipitation = max(0, np.random.normal(0.2, 0.5))
    daylight_factor = max(0, np.sin((hour - 6) / 12 * np.pi))
    solar_output = max(
        0,
        daylight_factor * (1 - cloud_cover / 120) * (temperature / 30) * 1000
    )
    solar_output += np.random.normal(0, 20)
    solar_output = max(0, solar_output)
    rows.append([
        ts.strftime('%Y-%m-%d %H:%M:%S'),
        temperature,
        humidity,
        cloud_cover,
        wind_speed,
        precipitation,
        solar_output
    ])

df = pd.DataFrame(rows, columns=[
    "timestamp", "temperature", "humidity", "cloud_cover",
    "wind_speed", "precipitation", "solar_output"
])

# Save raw dataset locally
df.to_csv("solar_weather_dataset.csv", index=False)
print("Dataset created: solar_weather_dataset.csv")
print("Total rows generated:", len(df))

Dataset created: solar_weather_dataset.csv
Total rows generated: 1440


### 2. Exploratory Data Analysis (EDA)
Check the data using `head()`, `info()`, `describe()`, and `isna().sum()`, then plot the first 3 days of solar output.

In [3]:
#Exploratory Data Analysis
raw_df = pd.read_csv("solar_weather_dataset.csv")

# 1. Preview the first 5 records
print("--- First 5 Rows ---")
print(raw_df.head())

# 2. Schema and data types
print("\n--- DataFrame Info ---")
raw_df.info()

# 3. Statistical summary
print("\n--- Summary Statistics ---")
print(raw_df.describe())

# 4. Check for missing values
print("\n--- Missing Values Count ---")
print(raw_df.isna().sum())

# 5. Quick line plot of first 3 days (first 72 hours)
sample_3days = raw_df.head(72)
fig_eda = px.line(
    sample_3days,
    x="timestamp",
    y="solar_output",
    title="Solar Power Output (First 3 Days: Jan 1 - Jan 3, 2024)",
    labels={"timestamp": "Timestamp", "solar_output": "Solar Output (kW)"}
)
fig_eda.show()

--- First 5 Rows ---
             timestamp  temperature   humidity  cloud_cover  wind_speed  \
0  2024-01-01 00:00:00    18.496714  58.617357    56.192213    8.046060   
1  2024-01-01 01:00:00    22.167403  67.674347    28.263140    6.085120   
2  2024-01-01 02:00:00    23.241962  40.867198     0.000000    3.875425   
3  2024-01-01 03:00:00    24.163044  45.876963    76.641219    4.548447   
4  2024-01-01 04:00:00    26.115871  61.109226    11.225161    5.751396   

   precipitation  solar_output  
0       0.082923      0.000000  
1       0.000000      0.000000  
2       0.000000      6.284947  
3       0.233764      0.000000  
4       0.000000      0.000000  

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      1440 non-null   object 
 1   temperature    1440 non-null   float64
 2   humidity       1440

### 3. Load and Clean Data
Load the CSV, parse timestamps, interpolate any missing values, and clip values to valid ranges (like humidity 0–100% and solar output >= 0).

In [4]:
#Load and clean data using preprocess.py
clean_df = load_and_clean_data("solar_weather_dataset.csv")
print("Clean dataset loaded. Shape:", clean_df.shape)
print("Total missing values remaining:", clean_df.isna().sum().sum())

Clean dataset loaded. Shape: (1440, 7)
Total missing values remaining: 0


### 4. Feature Engineering
We add simple time features:
- `hour`: Hour of the day (0 to 23).
- `sin_hour` & `cos_hour`: Cyclical hour using sin(2π × hour / 24) and cos(2π × hour / 24) so 23:00 and 00:00 connect smoothly.
- `day_of_week`: Day index (0 = Monday, ..., 6 = Sunday).
- `is_weekend`: 1 for Saturday/Sunday, 0 for weekdays.

In [5]:
#Feature engineering using preprocess.py
featured_df = add_features(clean_df)
print("Features created successfully. First 5 rows:")
print(featured_df[["timestamp", "hour", "sin_hour", "cos_hour", "day_of_week", "is_weekend"]].head())

Features created successfully. First 5 rows:
            timestamp  hour  sin_hour  cos_hour  day_of_week  is_weekend
0 2024-01-01 00:00:00     0  0.000000  1.000000            0           0
1 2024-01-01 01:00:00     1  0.258819  0.965926            0           0
2 2024-01-01 02:00:00     2  0.500000  0.866025            0           0
3 2024-01-01 03:00:00     3  0.707107  0.707107            0           0
4 2024-01-01 04:00:00     4  0.866025  0.500000            0           0


### 5. Chronological Train / Validation Split
Split the data chronologically (no shuffling) so the model only learns from past data:
- **Training Set:** First 50 days (50 × 24 = 1200 rows).
- **Validation Set:** Last 10 days (10 × 24 = 240 rows).

In [6]:
#Chronological Train / Validation Split
feature_cols = [
    "temperature", "humidity", "cloud_cover", "wind_speed", "precipitation",
    "hour", "sin_hour", "cos_hour", "day_of_week", "is_weekend"
]
target_col = "solar_output"

# Split index: 50 days * 24 hours = 1200 rows
split_idx = 50 * 24

train_df = featured_df.iloc[:split_idx]
val_df = featured_df.iloc[split_idx:]

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]

print(f"Training set:   {len(X_train)} hours (First 50 days)")
print(f"Validation set: {len(X_val)} hours (Last 10 days)")

Training set:   1200 hours (First 50 days)
Validation set: 240 hours (Last 10 days)


### 6. Model Training, Evaluation, and Selection
We call `train_and_save_model()` from `train.py`:
1. **Baseline Model:** `LinearRegression`.
2. **Main Model:** `RandomForestRegressor` tuned using a simple loop over `n_estimators` (50, 100, 150) and `max_depth` (5, 10, None).
3. Compares MAE, RMSE, and R2 on the validation set, picks the model with the lowest RMSE, and saves it to disk (`best_solar_model.pkl`).

In [7]:
#Train models and save best artifact using train.py
best_model, metrics_df = train_and_save_model(X_train, y_train, X_val, y_val, model_path="best_solar_model.pkl")

=== Model Performance on Validation Set ===
                      Model n_estimators max_depth   MAE  RMSE     R2
LinearRegression (Baseline)            -         - 59.14 79.81 0.7707
      RandomForestRegressor           50         5 18.38 26.50 0.9747
      RandomForestRegressor           50        10 15.27 20.87 0.9843
      RandomForestRegressor           50      None 15.46 20.98 0.9842
      RandomForestRegressor          100         5 18.38 26.67 0.9744
      RandomForestRegressor          100        10 15.24 20.83 0.9844
      RandomForestRegressor          100      None 15.46 20.98 0.9842
      RandomForestRegressor          150         5 18.39 26.65 0.9744
      RandomForestRegressor          150        10 15.04 20.54 0.9848
      RandomForestRegressor          150      None 15.33 20.69 0.9846

Best Model (n_estimators=150, max_depth=10) saved to 'best_solar_model.pkl'


### 7. Weather Forecast API (Pune)
Fetch the next 48 hours of weather for Pune (Lat 18.52, Lon 73.86) from Open-Meteo via `get_pune_weather_forecast()` in `weather_api.py`.
- Free public API with no API key needed.
- Requests wind speed in m/s (`wind_speed_unit=ms`).
- Retries up to 3 times (with a 2-second pause, handling rate-limit 429).
- If unreachable, uses fallback data so the code does not crash.

In [8]:
#Fetch 48-hour Pune weather forecast using weather_api.py
forecast_raw, is_fallback = get_pune_weather_forecast()
print("Forecast DataFrame shape:", forecast_raw.shape)
print("Is fallback data used?:", is_fallback)
print(forecast_raw.head())

Successfully fetched live 48-hour forecast from Open-Meteo API.
Forecast DataFrame shape: (48, 6)
Is fallback data used?: False
            timestamp  temperature  humidity  cloud_cover  wind_speed  \
0 2026-11-18 16:00:00         29.2        31            0        3.69   
1 2026-11-18 17:00:00         28.1        34            0        4.04   
2 2026-11-18 18:00:00         26.9        37            0        2.80   
3 2026-11-18 19:00:00         26.5        37            3        3.08   
4 2026-11-18 20:00:00         25.0        42            1        2.91   

   precipitation  
0            0.0  
1            0.0  
2            0.0  
3            0.0  
4            0.0  


### 8. Predict Solar Output on Forecast Data
Load the saved model (`best_solar_model.pkl`), add the same features to the forecast data, predict solar output, and clip negative values to 0.

In [9]:
# Generate predictions
# Load saved model artifact
saved_model = joblib.load("best_solar_model.pkl")

# Apply the exact same feature engineering function
forecast_featured = add_features(forecast_raw)

# Extract matching features
X_future = forecast_featured[feature_cols]

# Predict future solar output and clip negatives to 0
predictions = saved_model.predict(X_future)
forecast_featured["predicted_solar_output"] = np.clip(predictions, 0, None).round(2)

print("48-Hour solar power predictions generated successfully!")

48-Hour solar power predictions generated successfully!


### 9. Prediction Table (Next 24 Hours)
Table showing the first 24 hours of predictions with weather conditions.

In [10]:
#Display first 24 hours of predictions
display_cols = ["timestamp", "temperature", "cloud_cover", "humidity", "predicted_solar_output"]
pred_table_24h = forecast_featured[display_cols].head(24)

print("=== 24-Hour Solar Power Predictions (Pune, India) ===")
print(pred_table_24h.to_string(index=False))

=== 24-Hour Solar Power Predictions (Pune, India) ===
          timestamp  temperature  cloud_cover  humidity  predicted_solar_output
2026-11-18 16:00:00         29.2            0        31                  559.17
2026-11-18 17:00:00         28.1            0        34                  135.11
2026-11-18 18:00:00         26.9            0        37                    7.95
2026-11-18 19:00:00         26.5            3        37                    7.97
2026-11-18 20:00:00         25.0            1        42                   11.77
2026-11-18 21:00:00         23.8            1        45                   11.89
2026-11-18 22:00:00         22.9            0        48                    7.64
2026-11-18 23:00:00         22.1            1        51                    9.49
2026-11-19 00:00:00         21.3            0        55                    9.00
2026-11-19 01:00:00         21.0            1        56                    7.38
2026-11-19 02:00:00         20.5            0        57           

> **Note on Timestamp Alignment:**  
> The dataset is from early 2024, but the forecast is for today. We shift the historical data forward by whole days so it ends exactly 1 hour before the forecast starts. Shifting by whole days preserves the hour of the day.

In [11]:
# Plotly interactive visualization
# 1. Align historical end hour to exactly one hour prior to forecast start
forecast_start = forecast_featured["timestamp"].iloc[0]
last_hour = (forecast_start - pd.Timedelta(hours=1)).hour
hist_end = featured_df[featured_df["timestamp"].dt.hour == last_hour]["timestamp"].iloc[-1]
history_7d = featured_df[featured_df["timestamp"] <= hist_end].tail(7 * 24).copy()
days_to_shift = ((forecast_start - pd.Timedelta(hours=1)).normalize() - hist_end.normalize()).days
history_7d["shifted_timestamp"] = history_7d["timestamp"] + pd.Timedelta(days=days_to_shift)

# 2. Build interactive Plotly chart
fig = go.Figure()

# Historical Solar Output trace
fig.add_trace(go.Scatter(
    x=history_7d["shifted_timestamp"],
    y=history_7d["solar_output"],
    mode="lines",
    name="Historical Solar Output (Past 7 Days)",
    line=dict(color="#1f77b4", width=2)
))

# Forecast Solar Output trace
fig.add_trace(go.Scatter(
    x=forecast_featured["timestamp"],
    y=forecast_featured["predicted_solar_output"],
    mode="lines+markers",
    name="Predicted Solar Output (Next 48 Hours)",
    line=dict(color="#ff7f0e", width=2.5, dash="dash"),
    marker=dict(size=4)
))

# Chart title (includes fallback warning if fallback data is active)
chart_title = "<b>Solar Output: Past 7 Days + Next 48 Hours (Pune)</b>"
if is_fallback:
    chart_title += " (fallback data)"

# Layout formatting
fig.update_layout(
    title=chart_title,
    xaxis_title="Timestamp",
    yaxis_title="Solar Power Output (kW)",
    template="plotly_white",
    hovermode="x unified",
    margin=dict(t=100),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=550
)

# Display chart and export standalone HTML
fig.show()
fig.write_html("forecast_chart.html")
print("Interactive chart exported to 'forecast_chart.html'")

Interactive chart exported to 'forecast_chart.html'


### 11. Pipeline Validation Tests
Run 3 simple assert tests to check feature columns, non-negative predictions, and the 48-hour output length.

In [12]:
# Assert tests
# Test 1: Verify all 10 expected feature columns exist in forecast features
expected_cols = [
    "temperature", "humidity", "cloud_cover", "wind_speed", "precipitation",
    "hour", "sin_hour", "cos_hour", "day_of_week", "is_weekend"
]
for col in expected_cols:
    assert col in forecast_featured.columns, f"Test 1 Failed: Column '{col}' is missing!"
print("[PASS] Test 1 Passed: All 10 expected feature columns are present.")

# Test 2: Verify predictions are strictly non-negative
assert (forecast_featured["predicted_solar_output"] >= 0).all(), "Test 2 Failed: Found negative predictions!"
print("[PASS] Test 2 Passed: All predictions are non-negative (clipped to 0).")

# Test 3: Verify forecast output contains 48 hourly predictions
assert len(forecast_featured) == 48, f"Test 3 Failed: Expected 48 rows, got {len(forecast_featured)}"
print("[PASS] Test 3 Passed: Forecast output contains exactly 48 hourly predictions.")

[PASS] Test 1 Passed: All 10 expected feature columns are present.
[PASS] Test 2 Passed: All predictions are non-negative (clipped to 0).
[PASS] Test 3 Passed: Forecast output contains exactly 48 hourly predictions.


### 12. Conclusion, Assumptions, and Next Steps

#### Assumptions:
1. **Night solar power:** Solar generation is near 0 at night (the dataset's noise gives about 8 kW on average).
2. **Timezone:** The API is requested with `timezone=Asia/Kolkata` and compared with the computer’s current time, so the code assumes the computer is set to Indian time (IST). The dataset timestamps are treated as local time too.
3. **Missing forecast fields:** If an API field is missing, a default value is used (like 25 °C and 50% humidity), and empty spots in the lists are filled from nearby hours with interpolation.
4. **History shift for chart:** Shifting history forward by whole days keeps the same hour of the day.
5. **Valid ranges:** Values are clipped to natural limits (humidity 0–100%, non-negative power).

#### What I Would Improve Next:
1. Collect data across all seasons (summer and monsoon) to see how clouds and extreme heat affect the panels.
2. Try other simple models like XGBoost to compare speed and accuracy.
3. Build a small web UI with Streamlit so anyone can check the forecast easily.

In [13]:
#Completion status
print("=" * 60)
print("Solar Power Prediction Pipeline completed successfully!")
print("Model artifact: best_solar_model.pkl")
print("Interactive Plotly chart and 48-hour forecast generated.")
print("=" * 60)

Solar Power Prediction Pipeline completed successfully!
Model artifact: best_solar_model.pkl
Interactive Plotly chart and 48-hour forecast generated.
